# Netflix Rating Prediction Using Collaborative Filtering

In this notebook, I am going to use two CF approaches to estimate missing values in a Netflix rating dataset:
1. Singular Value Decomposition (SVD)
2. Nonnegative Matrix Factorization (NMF)

### **Goal: Fill missing entries in the user-item rating matrix and provide personalized movie recommendations**


In [1]:
import numpy as np
import pandas as pd

np.random.seed(40) # for reproducibility 

## Load and Inspect Data 
- `netflix_rating.csv`: User-item rating matrix with User_ID as rows and Movie_IDs as columns
- `netflix_movie.csv`: Movie data (ID, Year, Name)

In [2]:
rating_data = pd.read_csv("netflix_rating.csv", index_col="User_ID") # index_col argument allows us to index the df by user_id instead of traditional indices 
movie_data = pd.read_csv("netflix_movie.csv")

print("Ratings Dataset")
print(f"Shape: {rating_data.shape}")
print(f"Users: {rating_data.shape[0]}, Items: {rating_data.shape[1]}")

print("\nMovie Dataset")
print(f"Number of movies: {len(movie_data)}")
print("\nFirst 10 movies:")
print(movie_data.head(10))


Ratings Dataset
Shape: (14328, 150)
Users: 14328, Items: 150

Movie Dataset
Number of movies: 17770

First 10 movies:
   Movie_ID  Year                          Name
0         1  2003               Dinosaur Planet
1         2  2004    Isle of Man TT 2004 Review
2         3  1997                     Character
3         4  1994  Paula Abdul's Get Up & Dance
4         5  2004      The Rise and Fall of ECW
5         6  1997                          Sick
6         7  1992                         8 Man
7         8  2004    What the #$*! Do We Know!?
8         9  1991      Class of Nuke 'Em High 2
9        10  2001                       Fighter


## Preparing Data 

In the following few cells we will:

- convert rating dataset into a numpy array called R
- create a mask to indicate known data 
- fill missing values with column means

In [3]:
# Convert to numpy array
R = rating_data.to_numpy(dtype = float) # User-item rating matrix

# Extract user ids and movie ids for later 
user_ids = rating_data.index.values
movie_ids = rating_data.columns.astype(int).values

# Create a mask for known ratings (True when known, False otherwise)
known_mask = ~np.isnan(R) 

We can see below that R is extremely sparse

In [4]:
num_known = np.sum(known_mask)
num_total = R.size
sparsity = 1 - (num_known / num_total)

print(f"Known ratings: {num_known:,}")
print(f"Total entries: {num_total:,}")
print(f"Sparsity: {sparsity:.2%}")

Known ratings: 170,814
Total entries: 2,149,200
Sparsity: 92.05%


### Simple Imputation
As mentioned before, we will fill the missing values in R with the column means

In [5]:
col_avg = np.nanmean(R, axis=0) # along axis 0 (rows)
R_filled = R.copy()
for j in range(R.shape[1]):
    R_filled[np.isnan(R[:,j]),j] = col_avg[j]
print(f"R_filled:\n {R_filled}")

R_filled:
 [[3.78205128 3.18907104 3.16888889 ... 3.90909091 3.68062827 3.87537538]
 [3.78205128 5.         3.16888889 ... 3.90909091 3.68062827 3.87537538]
 [3.78205128 3.18907104 3.16888889 ... 3.90909091 3.68062827 3.87537538]
 ...
 [3.78205128 3.18907104 3.16888889 ... 3.90909091 3.68062827 3.87537538]
 [3.78205128 4.         3.16888889 ... 3.90909091 3.68062827 3.87537538]
 [3.78205128 3.18907104 3.16888889 ... 3.90909091 3.68062827 3.87537538]]


## Collaborative Filtering Using SVD

- $U$: User-feature matrix ($m \times k$)  
- $\Sigma$: Diagonal matrix of singular values ($k \times k$)  
- $V$: Item-feature matrix ($k \times n$)  

We will use $k = 15$ features to create a low-rank approximation

In [6]:
def svd_collaborative_filtering(R, k=15):
    """
    Perform SVD-based collaborative filtering

    Parameters:
    R: User-item rating matrix (with col means for missing values)
    k: Number of latent features
    
    Returns:
    R_pred: Predicted rating matrix
    """
    col_means = np.mean(R, axis=0)
    R_centered = R - col_means

    # SVD
    U, s, Vt = np.linalg.svd(R_centered, full_matrices=False)

    # Truncate
    U_k = U[:, :k]
    s_k = s[:k]
    Vt_k = Vt[:k, :]

    # Reconstruct
    R_pred = U_k @ np.diag(s_k) @ Vt_k + col_means

    return R_pred

# Apply SVD
R_svd = svd_collaborative_filtering(R_filled, k=15)


## Collaborative Filtering Using NMF

Non-Negative Matrix Factorization decomposes $R$ as:

$$
R \approx P Q
$$

where:
- $P$: User feature matrix ($m \times k$)  
- $Q$: Item feature matrix ($k \times n$)  

### Gradients

$$
\nabla_P J = -Y Q^T = -(\nabla_Y J) Q^T
$$

$$
\nabla_Q J = -P^T Y = -P^T (\nabla_Y J)
$$

### Update Rules (multiplication ensures nonnegativity)

$$
P = P \odot \frac{R Q^T}{P Q Q^T + \epsilon}
$$

$$
Q = Q \odot \frac{P^T R}{P^T P Q + \epsilon}
$$

In [7]:
def initialize_nmf_factors(m, n, k=15):
    """
    Initialize P and Q matrices with non-negative random values.
    
    Parameters:
    m: Number of users
    n: Number of items
    k: Number of latent features
    
    Returns:
    P: User feature matrix
    Q: Item feature matrix
    """
    P = np.random.rand(m, k) + 1e-3
    Q = np.random.rand(k, n) + 1e-3 # avoids dividing by zero 
    return P, Q

In [8]:
def compute_nmf_rmse(R, P, Q, mask):
    """
    Compute error on known ratings (matching reference approach).
    """
    R_pred = P @ Q
    error = np.sqrt(np.mean((R_pred[mask] - R[mask]) ** 2))
    return error 

In [9]:
def fit_nmf(R, mask, k=15, max_iterations=500, error_threshold=0.4, epsilon=1e-12):
    """
    Fit NMF model using multiplicative update rules.
    
    Parameters:
    R: User-item rating matrix
    mask: True where ratings are known
    k: Number of latent features
    max_iterations: Maximum number of iterations
    error_threshold: Convergence goal (RMSE on known ratings)
    epsilon: Small value to avoid division by zero
    
    Returns:
    P: Fitted user feature matrix
    Q: Fitted item feature matrix
    errors: RMSE at each iteration
    """
    m, n = R.shape
    P, Q = initialize_nmf_factors(m, n, k)
    R_masked = R * mask

    errors = []

    for iteration in range(max_iterations):

        # Multiplicative update for P
        # P = P * (R Q^T) / (PQ Q^T + epsilon)
        numerator_P = R_masked @ Q.T
        denominator_P = ((P @ Q) * mask) @ Q.T + epsilon
        P *= (numerator_P / denominator_P)

        
        # Multiplicative update for Q
        # Q = Q * (P^T R) / (P^T PQ + epsilon)
        numerator_Q = P.T @ R_masked
        denominator_Q = P.T @ ((P @ Q) * mask) + epsilon
        Q *= (numerator_Q / denominator_Q)
        
        # Compute error
        rmse = compute_nmf_rmse(R, P, Q, mask)
        errors.append(rmse)
        
        # Check convergence
        if rmse < error_threshold:
            print(f"\nConverged at iteration {iteration + 1}")
            print(f"Final RMSE: {rmse:.6f}")
            break
    
    if iteration + 1 == max_iterations:
        print(f"Final RMSE: {rmse:.6f}")
    
    return P, Q, errors


In [10]:
def predict_nmf(P, Q):
    """
    Generate predictions for all entries.
    
    Parameters:
    P: User feature matrix
    Q: Item feature matrix
    
    Returns:
    R_pred: Predicted rating matrix
    """
    return P @ Q

## RMSE for NMF Method

Running enough iterations to ensure that the error on known ratings is below 0.4

In [11]:
P, Q, nmf_errors = fit_nmf(R_filled, known_mask, k=15, max_iterations=500, error_threshold=0.4)
R_nmf = predict_nmf(P, Q)



Converged at iteration 278
Final RMSE: 0.399894


## Comparing SVD vs NMF
Based on the metrics below, NMF is a significant improvement from the centered SVD method used prior 

In [14]:
# Clip predictions to valid rating range [1, 5]
R_svd_clipped = np.clip(R_svd, 1, 5)
R_nmf_clipped = np.clip(R_nmf, 1, 5)

# Compute RMSE on known ratings for both methods
svd_error = np.sqrt(np.mean((R[known_mask] - R_svd_clipped[known_mask]) ** 2))
nmf_error = np.sqrt(np.mean((R[known_mask] - R_nmf_clipped[known_mask]) ** 2))

print("SVD")
print(f"RMSE on known ratings: {svd_error:.6f}")

print("\nNMF")
print(f"RMSE on known ratings: {nmf_error:.6f}")

print(f"\nImprovement (NMF vs SVD): {((svd_error - nmf_error) / svd_error * 100):.2f}%")

SVD
RMSE on known ratings: 0.712498

NMF
RMSE on known ratings: 0.397192

Improvement (NMF vs SVD): 44.25%


## Movie Recommendations

For each user we recommend the top 2 movies they haven't rated yet based on predicted ratings

In [71]:
def get_recommendations(user_idx, R_pred, R_original, movie_data, rating_data, n_recommendations=2):
    """
    Get top N movie recommendations for a user
    
    Parameters:
    user_idx: Index of the user (row in rating matrix)
    R_pred: Predicted rating matrix
    R_original: Original rating matrix 
    movie_df: Movie dataframe
    n_recommendations: Number of recommendations to return
    
    Returns:
    recommendations: list of tuples [(movie_name, predicted_rating), ...]
    """
    user_predictions = R_pred[user_idx]
    user_ratings = R_original[user_idx]
    
    # Find unrated items (NaN in original)
    unrated_mask = np.isnan(user_ratings)
    
    # Get indices of unrated items
    unrated_indices = np.where(unrated_mask)[0]
    
    # Get predictions only for unrated items
    unrated_predictions = user_predictions[unrated_indices]
    
    # Get indices of top n = 2 highest predicitions 
    top_local_idx = np.argsort(unrated_predictions)[-n_recommendations:][::-1]
    
    recommendations = []
    for local_idx in top_local_idx:
        item_idx = unrated_indices[local_idx]
        movie_id = rating_data.columns[item_idx]
        predicted_rating = user_predictions[item_idx]
        
        # Get movie name
        movie_row = movie_data[movie_data['Movie_ID'] == int(movie_id)]
        if len(movie_row) > 0:
            movie_name = movie_row['Name'].values[0]
        else:
            movie_name = f"Movie ID {movie_id}"
        
        recommendations.append((movie_name, predicted_rating))
    
    return recommendations

print("Movie Reccomendations (Using NMF Predictions)")

# Get recommendations for first 10 users
for user_idx in range(min(10, len(user_ids))):
    user_id = user_ids[user_idx]
    
    recommendations = get_recommendations(user_idx, R_nmf_clipped, R, movie_data, rating_data, n_recommendations=2)
    
    print(f"\nUser {user_id} Recommendations:")

    rank = 1
    for movie_name, predicted_rating in recommendations:
        print(f"  {rank}. {movie_name} (Predicted Rating: {predicted_rating:.2f}/5.0)")
        rank += 1

Movie Reccomendations (Using NMF Predictions)

User 6 Recommendations:
  1. Viva La Bam: Season 1 (Predicted Rating: 5.00/5.0)
  2. Husbands and Wives (Predicted Rating: 5.00/5.0)

User 7 Recommendations:
  1. Kill Bill: Vol. 2 (Predicted Rating: 5.00/5.0)
  2. Herbie Rides Again (Predicted Rating: 5.00/5.0)

User 79 Recommendations:
  1. Invader Zim (Predicted Rating: 5.00/5.0)
  2. Michael Moore's The Awful Truth: Season 2 (Predicted Rating: 5.00/5.0)

User 97 Recommendations:
  1. Michael Moore's The Awful Truth: Season 2 (Predicted Rating: 5.00/5.0)
  2. Star Trek: Deep Space Nine: Season 5 (Predicted Rating: 5.00/5.0)

User 134 Recommendations:
  1. Kill Bill: Vol. 2 (Predicted Rating: 5.00/5.0)
  2. Absolutely Fabulous: Series 5 (Predicted Rating: 5.00/5.0)

User 169 Recommendations:
  1. Wings of Desire (Predicted Rating: 5.00/5.0)
  2. The Unsinkable Molly Brown (Predicted Rating: 5.00/5.0)

User 183 Recommendations:
  1. Jade (Predicted Rating: 4.75/5.0)
  2. Sex and the City:

## Saving the New Estimated Dataset!

In [73]:
# Clip predictions
R_svd_clipped = np.clip(R_svd, 1, 5)
R_nmf_clipped = np.clip(R_nmf, 1, 5)

# Keep originals fill only missing entries
R_svd_output = R_filled.copy()  # Start with original known + column mean fillins
R_svd_output[~known_mask] = R_svd_clipped[~known_mask]

R_nmf_output = R_filled.copy()
R_nmf_output[~known_mask] = R_nmf_clipped[~known_mask]

# Convert to DataFrames
R_svd_df = pd.DataFrame(R_svd_output, index=user_ids, columns=movie_ids)
R_nmf_df = pd.DataFrame(R_nmf_output, index=user_ids, columns=movie_ids)

R_svd_df.index.name = 'User_ID'
R_nmf_df.index.name = 'User_ID'

# Save to CSV
R_svd_df.to_csv('netflix_ratings_svd_filled.csv')
R_nmf_df.to_csv('netflix_ratings_nmf_filled.csv')

# Show sample
print(f"\nSample of NMF-filled ratings:")
print(R_nmf_df.iloc[:5, :10].round(2))


Sample of NMF-filled ratings:
           3     8     16    17    18    26    28    30    32    33
User_ID                                                            
6        2.19  1.55  2.84  2.98  2.85  2.68  1.23  3.00  4.43  5.00
7        4.02  5.00  4.55  3.28  5.00  2.32  4.00  5.00  5.00  5.00
79       1.24  4.14  3.64  2.73  2.63  2.49  3.73  3.00  3.00  3.61
97       5.00  5.00  5.00  1.71  3.74  3.20  5.00  3.55  4.72  5.00
134      5.00  2.91  5.00  3.37  5.00  3.13  5.00  5.00  5.00  3.97
